In [ ]:
  # PRAKTIKUM 3: MENILAI MESIN PENCARI BUATAN SENDIRI

# FUNGSI EVALUASI
def precision_at_k(hasil, relevan, k):
    return sum(1 for d in hasil[:k] if d in relevan) / k


def recall(hasil, relevan):
    cocok = sum(1 for d in hasil if d in relevan)
    return cocok / len(relevan)


def f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def average_precision(hasil, relevan):
    hit, total = 0, 0.0

    for i, d in enumerate(hasil, start=1):
        if d in relevan:
            hit += 1
            total += hit / i

    return total / len(relevan)

# QUERY 1
query1 = "love"

relevan1 = {2, 3, 5, 9}
hasil1 = [2, 7, 3, 1, 5]

p1 = precision_at_k(hasil1, relevan1, 5)
r1 = recall(hasil1, relevan1)

print("QUERY 1 :", query1)
print("Precision@5 =", round(p1, 4))
print("Recall      =", round(r1, 4))
print("F1          =", round(f1(p1, r1), 4))
print("AP          =", round(average_precision(hasil1, relevan1), 4))

# QUERY 2
query2 = "sad"

relevan2 = {1, 4, 6, 8}
hasil2 = [4, 2, 1, 7, 6]

p2 = precision_at_k(hasil2, relevan2, 5)
r2 = recall(hasil2, relevan2)

print("\nQUERY 2 :", query2)
print("Precision@5 =", round(p2, 4))
print("Recall      =", round(r2, 4))
print("F1          =", round(f1(p2, r2), 4))
print("AP          =", round(average_precision(hasil2, relevan2), 4))

# QUERY 3
query3 = "happy"

relevan3 = {3, 5, 7, 10}
hasil3 = [5, 3, 2, 8, 7]

p3 = precision_at_k(hasil3, relevan3, 5)
r3 = recall(hasil3, relevan3)

print("\nQUERY 3 :", query3)
print("Precision@5 =", round(p3, 4))
print("Recall      =", round(r3, 4))
print("F1          =", round(f1(p3, r3), 4))
print("AP          =", round(average_precision(hasil3, relevan3), 4))

# MAP (MEAN AVERAGE PRECISION)
ap1 = average_precision(hasil1, relevan1)
ap2 = average_precision(hasil2, relevan2)
ap3 = average_precision(hasil3, relevan3)

map_score = (ap1 + ap2 + ap3) / 3

print("\n============================================")
print("MAP =", round(map_score, 4))
print("============================================")

QUERY 1 : love
Precision@5 = 0.6
Recall      = 0.75
F1          = 0.6667
AP          = 0.5667

QUERY 2 : sad
Precision@5 = 0.6
Recall      = 0.75
F1          = 0.6667
AP          = 0.5667

QUERY 3 : happy
Precision@5 = 0.6
Recall      = 0.75
F1          = 0.6667
AP          = 0.65

MAP = 0.5944


In [ ]:
import math
import re
from collections import Counter

# ================== DATA 10 JUDUL SKRIPSI ==================
koleksi_dokumen = {
    1: "Sistem Kontrol dan Monitoring Konsumsi Daya Listrik Penerangan Rumah Tangga Berbasis Android",
    2: "Prototipe Sistem Pengering Sepatu Otomatis Berbasis Internet of Things ( IoT )",
    3: "Analisis Kualitas Layanan Jaringan Wireless LAN di Lingkungan Jurusan Teknik Informatika dan Komputer Menggunakan Parameter QoS",
    4: "Smart Home Monitoring Pagar Rumah Menggunakan Pengenalan Wajah Berbasis Internet of Things ( IoT )",
    5: "Pengembangan Sistem Informasi Akademik Berbasis Website di SMP Negeri 6 Bangkala Barat Kabupaten Jeneponto",
    6: "Sistem Keamanan Jaringan terhadap Serangan Packet Sniffing Berbasis Honeypot",
    7: "Analisis Efektivitas IPTables dalam Melindungi Jaringan dari Serangan DDoS",
    8: "Analisis Kualitas Pembelajaran Daring di Jurusan Teknik Komputer dan Jaringan SMKN 1 Bone",
    9: "Pengembangan Sistem Informasi Sekolah Berbasis Web Darul Ulum Ageng Maros",
    10: "Pengaruh Penerapan Project Based Learning terhadap Kesiapan Kerja Siswa SMK Negeri 2 Makassar"
}

# ================== STOPWORDS ==================
stopwords = {
    "yang", "di", "dan", "ke", "dari", "pada", "untuk",
    "dengan", "sebagai", "dalam", "ini", "itu", "oleh",
    "serta", "bagi", "para", "tersebut", "adalah",
    "merupakan", "pengaruh", "terhadap", "siswa",
    "mahasiswa", "belajar", "komputer", "informatika",
    "teknik", "pendidikan", "negeri", "universitas",
    "smp", "sma", "smk", "mts"
}

# ================== PREPROCESSING ==================
def preprocessing(teks):
    teks = teks.lower()
    teks = re.sub(r'[^a-zA-Z\s]', '', teks)
    tokens = teks.split()
    tokens = [word for word in tokens if word not in stopwords and len(word) > 2]
    return tokens

# ================== TF-IDF ==================
def build_tfidf(koleksi):
    doc_tokens = {}
    all_tokens = []

    for doc_id, doc in koleksi.items():
        tokens = preprocessing(doc)
        doc_tokens[doc_id] = tokens
        all_tokens.extend(tokens)

    df = Counter(all_tokens)
    N = len(koleksi)

    idf = {
        word: math.log(N / df[word])
        for word in df
    }

    tfidf_vectors = {}

    for doc_id, tokens in doc_tokens.items():
        tf = Counter(tokens)
        vector = {}

        for word, count in tf.items():
            vector[word] = count * idf[word]

        tfidf_vectors[doc_id] = vector

    return tfidf_vectors, idf


def vectorize_query(query, idf):
    tokens = preprocessing(query)
    tf = Counter(tokens)
    vector = {}

    for word, count in tf.items():
        if word in idf:
            vector[word] = count * idf[word]

    return vector


def cosine_similarity(vec1, vec2):
    intersection = set(vec1.keys()) & set(vec2.keys())

    dot = sum(
        vec1[word] * vec2[word]
        for word in intersection
    )

    norm1 = math.sqrt(
        sum(v**2 for v in vec1.values())
    )

    norm2 = math.sqrt(
        sum(v**2 for v in vec2.values())
    )

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot / (norm1 * norm2)

# ================== FUNGSI PENCARIAN ==================
def cari_dokumen(query, koleksi, tfidf_vectors, idf):

    query_vec = vectorize_query(query, idf)

    hasil = []

    for doc_id, doc_vec in tfidf_vectors.items():

        skor = cosine_similarity(
            query_vec,
            doc_vec
        )

        if skor > 0:
            hasil.append(
                (doc_id, skor, koleksi[doc_id])
            )

    hasil.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return hasil


def tampilkan(
    query,
    koleksi,
    tfidf_vectors,
    idf,
    k=3
):

    print(f"\nQuery: '{query}'")

    hasil = cari_dokumen(
        query,
        koleksi,
        tfidf_vectors,
        idf
    )

    if not hasil:
        print("  (tidak ada dokumen yang cocok)")
        return []

    else:
        top_k = hasil[:k]

        for i, (doc_id, skor, doc) in enumerate(
            top_k,
            start=1
        ):

            print(
                f"  Peringkat {i} | "
                f"Dokumen {doc_id} | "
                f"Skor: {skor:.4f}"
            )

            print(
                f"    {doc}\n"
            )

        return [
            doc_id
            for doc_id, _, _ in top_k
        ]

# ================== FUNGSI METRIK EVALUASI ==================
def precision_at_k(hasil, relevan, k):

    if k == 0:
        return 0

    return sum(
        1 for d in hasil[:k]
        if d in relevan
    ) / k


def recall(hasil, relevan):

    if len(relevan) == 0:
        return 0

    return sum(
        1 for d in hasil
        if d in relevan
    ) / len(relevan)


def f1_score(p, r):

    if p + r == 0:
        return 0

    return 2 * p * r / (p + r)


def average_precision(hasil, relevan):

    if len(relevan) == 0:
        return 0

    hit = 0
    total = 0.0

    for i, d in enumerate(
        hasil,
        start=1
    ):

        if d in relevan:

            hit += 1

            total += hit / i

    return total / len(relevan)

# ================== BANGUN TF-IDF ==================
tfidf_vectors, idf = build_tfidf(
    koleksi_dokumen
)

print(
    "Bagian 1 selesai! "
    "Data dan fungsi sudah siap."
)

# ================== GROUND TRUTH ==================
ground_truth = {

    "AI": [4],

    "Jaringan": [3, 6, 7, 8],

    "IoT": [2, 4]
}

# ================== MENAMPILKAN HASIL PENCARIAN ==================
print("\n==========================================")
print("HASIL PENCARIAN")
print("==========================================")

hasil_AI = tampilkan(
    "AI",
    koleksi_dokumen,
    tfidf_vectors,
    idf,
    k=3
)

hasil_Jaringan = tampilkan(
    "Jaringan",
    koleksi_dokumen,
    tfidf_vectors,
    idf,
    k=3
)

hasil_IoT = tampilkan(
    "IoT",
    koleksi_dokumen,
    tfidf_vectors,
    idf,
    k=3
)

# ================== EVALUASI ==================
def evaluasi_query(
    query,
    hasil,
    relevan
):

    p3 = precision_at_k(
        hasil,
        relevan,
        3
    )

    r = recall(
        hasil,
        relevan
    )

    f1 = f1_score(
        p3,
        r
    )

    ap = average_precision(
        hasil,
        relevan
    )

    print("\n------------------------------------------")
    print("Query :", query)
    print("Ground Truth :", relevan)
    print("Hasil Pencarian :", hasil)
    print("Precision@3 :", round(p3, 4))
    print("Recall :", round(r, 4))
    print("F1-Score :", round(f1, 4))
    print("AP :", round(ap, 4))

    return ap


# ================== HITUNG AP SETIAP QUERY ==================
ap_AI = evaluasi_query(
    "AI",
    hasil_AI,
    ground_truth["AI"]
)

ap_Jaringan = evaluasi_query(
    "Jaringan",
    hasil_Jaringan,
    ground_truth["Jaringan"]
)

ap_IoT = evaluasi_query(
    "IoT",
    hasil_IoT,
    ground_truth["IoT"]
)

# ================== HITUNG MAP ==================
MAP = (
    ap_AI +
    ap_Jaringan +
    ap_IoT
) / 3

print("\n==========================================")
print("HASIL AKHIR")
print("==========================================")
print("MAP :", round(MAP, 4))

Bagian 1 selesai! Data dan fungsi sudah siap.

HASIL PENCARIAN

Query: 'AI'
  (tidak ada dokumen yang cocok)

Query: 'Jaringan'
  Peringkat 1 | Dokumen 6 | Skor: 0.1819
    Sistem Keamanan Jaringan terhadap Serangan Packet Sniffing Berbasis Honeypot

  Peringkat 2 | Dokumen 7 | Skor: 0.1794
    Analisis Efektivitas IPTables dalam Melindungi Jaringan dari Serangan DDoS

  Peringkat 3 | Dokumen 8 | Skor: 0.1711
    Analisis Kualitas Pembelajaran Daring di Jurusan Teknik Komputer dan Jaringan SMKN 1 Bone


Query: 'IoT'
  Peringkat 1 | Dokumen 2 | Skor: 0.2952
    Prototipe Sistem Pengering Sepatu Otomatis Berbasis Internet of Things ( IoT )

  Peringkat 2 | Dokumen 4 | Skor: 0.2474
    Smart Home Monitoring Pagar Rumah Menggunakan Pengenalan Wajah Berbasis Internet of Things ( IoT )


------------------------------------------
Query : AI
Ground Truth : [4]
Hasil Pencarian : []
Precision@3 : 0.0
Recall : 0.0
F1-Score : 0
AP : 0.0

------------------------------------------
Query : Jaringan

In [ ]:
print("="*60)
print("QUERY: 'AI'")
print("="*60)

query = "AI"
ground_truth = {4}  # Dokumen 4 relevan dengan AI

# Cari dokumen
hasil = cari_dokumen(
    query,
    koleksi_dokumen,
    tfidf_vectors,
    idf
)

# Tampilkan 3 teratas
print("\n--- HASIL PENCARIAN 3 TERATAS ---")

if not hasil:
    print("  (tidak ada dokumen yang cocok)")
    top3_ids = []

else:
    top3_ids = []

    for i, (doc_id, skor, doc) in enumerate(
        hasil[:3],
        start=1
    ):

        print(
            f"  Peringkat {i} | "
            f"Dokumen {doc_id} | "
            f"Skor: {skor:.4f}"
        )

        print(
            f"    {doc[:100]}..."
        )

        top3_ids.append(doc_id)

# Hitung metrik
p3 = precision_at_k(
    top3_ids,
    ground_truth,
    3
)

r = recall(
    top3_ids,
    ground_truth
)

f1 = f1_score(
    p3,
    r
)

ap = average_precision(
    top3_ids,
    ground_truth
)

print("\n--- METRIK EVALUASI ---")

print(
    f"  Ground Truth : {ground_truth}"
)

print(
    f"  Hasil 3 Top  : {top3_ids}"
)

print(
    f"  Precision@3 : {p3:.4f}"
)

print(
    f"  Recall      : {r:.4f}"
    if r is not None
    else "  Recall      : None"
)

print(
    f"  F1-Score    : {f1:.4f}"
    if f1 is not None
    else "  F1-Score    : None"
)

print(
    f"  AP          : {ap:.4f}"
)

QUERY: 'AI'

--- HASIL PENCARIAN 3 TERATAS ---
  (tidak ada dokumen yang cocok)

--- METRIK EVALUASI ---
  Ground Truth : {4}
  Hasil 3 Top  : []
  Precision@3 : 0.0000
  Recall      : 0.0000
  F1-Score    : 0.0000
  AP          : 0.0000


In [ ]:
print("="*60)
print("QUERY: 'Jaringan'")
print("="*60)

query = "Jaringan"
ground_truth = {3, 4, 9}  # Judul 3, 4, 9 relevan

# Cari dokumen
hasil = cari_dokumen(query, koleksi_dokumen, tfidf_vectors, idf)

# Tampilkan 3 teratas
print("\n--- HASIL PENCARIAN 3 TERATAS ---")
if not hasil:
    print("  (tidak ada dokumen yang cocok)")
    top3_ids = []
else:
    top3_ids = []
    for i, (doc_id, skor, doc) in enumerate(hasil[:3], start=1):
        print(f"  Peringkat {i} | Dokumen {doc_id} | Skor: {skor:.4f}")
        print(f"    {doc[:100]}...")
        top3_ids.append(doc_id)

# Hitung metrik
p3 = precision_at_k(top3_ids, ground_truth, 3)
r = recall(top3_ids, ground_truth)
f1 = f1_score(p3, r)
ap = average_precision(top3_ids, ground_truth)

print("\n--- METRIK EVALUASI ---")
print(f"  Ground Truth : {ground_truth}")
print(f"  Hasil 3 Top  : {top3_ids}")
print(f"  Precision@3 : {p3:.4f}")
print(f"  Recall      : {r:.4f}" if r is not None else "  Recall      : None")
print(f"  F1-Score    : {f1:.4f}" if f1 is not None else "  F1-Score    : None")
print(f"  AP          : {ap:.4f}")

QUERY: 'Jaringan'

--- HASIL PENCARIAN 3 TERATAS ---
  Peringkat 1 | Dokumen 6 | Skor: 0.1819
    Sistem Keamanan Jaringan terhadap Serangan Packet Sniffing Berbasis Honeypot...
  Peringkat 2 | Dokumen 7 | Skor: 0.1794
    Analisis Efektivitas IPTables dalam Melindungi Jaringan dari Serangan DDoS...
  Peringkat 3 | Dokumen 8 | Skor: 0.1711
    Analisis Kualitas Pembelajaran Daring di Jurusan Teknik Komputer dan Jaringan SMKN 1 Bone...

--- METRIK EVALUASI ---
  Ground Truth : {9, 3, 4}
  Hasil 3 Top  : [6, 7, 8]
  Precision@3 : 0.0000
  Recall      : 0.0000
  F1-Score    : 0.0000
  AP          : 0.0000


In [ ]:
print("="*60)
print("QUERY: 'IoT'")
print("="*60)

query = "IoT"
ground_truth = {2, 4}  # Judul 2 dan 4 relevan

# Cari dokumen
hasil = cari_dokumen(
    query,
    koleksi_dokumen,
    tfidf_vectors,
    idf
)

# Tampilkan 3 teratas
print("\n--- HASIL PENCARIAN 3 TERATAS ---")

if not hasil:
    print("  (tidak ada dokumen yang cocok)")
    top3_ids = []

else:
    top3_ids = []

    for i, (doc_id, skor, doc) in enumerate(
        hasil[:3],
        start=1
    ):

        print(
            f"  Peringkat {i} | "
            f"Dokumen {doc_id} | "
            f"Skor: {skor:.4f}"
        )

        print(
            f"    {doc[:100]}..."
        )

        top3_ids.append(doc_id)

# Hitung metrik
p3 = precision_at_k(
    top3_ids,
    ground_truth,
    3
)

r = recall(
    top3_ids,
    ground_truth
)

f1 = f1_score(
    p3,
    r
)

ap = average_precision(
    top3_ids,
    ground_truth
)

print("\n--- METRIK EVALUASI ---")

print(
    f"  Ground Truth : {ground_truth}"
)

print(
    f"  Hasil 3 Top  : {top3_ids}"
)

print(
    f"  Precision@3 : {p3:.4f}"
)

print(
    f"  Recall      : {r:.4f}"
    if r is not None
    else "  Recall      : None"
)

print(
    f"  F1-Score    : {f1:.4f}"
    if f1 is not None
    else "  F1-Score    : None"
)

print(
    f"  AP          : {ap:.4f}"
)

QUERY: 'IoT'

--- HASIL PENCARIAN 3 TERATAS ---
  Peringkat 1 | Dokumen 2 | Skor: 0.2952
    Prototipe Sistem Pengering Sepatu Otomatis Berbasis Internet of Things ( IoT )...
  Peringkat 2 | Dokumen 4 | Skor: 0.2474
    Smart Home Monitoring Pagar Rumah Menggunakan Pengenalan Wajah Berbasis Internet of Things ( IoT )...

--- METRIK EVALUASI ---
  Ground Truth : {2, 4}
  Hasil 3 Top  : [2, 4]
  Precision@3 : 0.6667
  Recall      : 1.0000
  F1-Score    : 0.8000
  AP          : 1.0000


In [ ]:
print("="*60)
print("MEAN AVERAGE PRECISION (MAP)")
print("="*60)

# Hitung hasil pencarian untuk setiap query
hasil_ai = cari_dokumen(
    "AI",
    koleksi_dokumen,
    tfidf_vectors,
    idf
)

hasil_jaringan = cari_dokumen(
    "Jaringan",
    koleksi_dokumen,
    tfidf_vectors,
    idf
)

hasil_iot = cari_dokumen(
    "IoT",
    koleksi_dokumen,
    tfidf_vectors,
    idf
)

# Ambil ID dokumen hasil pencarian
hasil_ai_ids = [
    doc_id for doc_id, _, _ in hasil_ai
]

hasil_jaringan_ids = [
    doc_id for doc_id, _, _ in hasil_jaringan
]

hasil_iot_ids = [
    doc_id for doc_id, _, _ in hasil_iot
]

# Ground Truth
ground_truth_ai = {4}
ground_truth_jaringan = {3, 6, 7, 8}
ground_truth_iot = {2, 4}

# Hitung AP
ap_ai = average_precision(
    hasil_ai_ids,
    ground_truth_ai
)

ap_jaringan = average_precision(
    hasil_jaringan_ids,
    ground_truth_jaringan
)

ap_iot = average_precision(
    hasil_iot_ids,
    ground_truth_iot
)

# Hitung MAP
ap_values = [
    ap_ai,
    ap_jaringan,
    ap_iot
]

map_score = sum(ap_values) / len(ap_values)

# Tampilkan hasil
print("AP untuk setiap kueri:")
print(f"  AI       : {ap_ai:.4f}")
print(f"  Jaringan : {ap_jaringan:.4f}")
print(f"  IoT      : {ap_iot:.4f}")

print(
    f"\nMAP = ({ap_ai:.4f} + "
    f"{ap_jaringan:.4f} + "
    f"{ap_iot:.4f}) / 3"
)

print(f"MAP = {map_score:.4f}")

MEAN AVERAGE PRECISION (MAP)
AP untuk setiap kueri:
  AI       : 0.0000
  Jaringan : 1.0000
  IoT      : 1.0000

MAP = (0.0000 + 1.0000 + 1.0000) / 3
MAP = 0.6667
